In [32]:
import pandas as pd
from pathlib import Path
import numpy as np
from datetime import datetime
BASE_DIR = Path.cwd()
RAW_DATA_PATH = BASE_DIR.parent / "DataSet" / "Raw" / "data.csv"
PROCESSED_DIR = BASE_DIR.parent / "DataSet" / "processed"
PROCESSED_DATA_PATH = PROCESSED_DIR / "ready_to_use_data.csv"

In [6]:
def fix_numeric_column(series):
    fixed = (
        series.astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)   # european comma -> dot
        .str.extract(r"([-+]?\d+(?:\.\d+)?)", expand=False)
    )
    return pd.to_numeric(fixed, errors="coerce")

In [24]:
def inspect_data(df):
    print(f"Number of rows: {len(df)}")
    print(f"Number of columns: {len(df.columns)}")

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    print(pd.DataFrame({"count": missing, "percent": missing_pct}))

    print("\nUnique values per column:")
    print(df.nunique())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nNumeric columns - basic stats:")
    print(df.describe())

    print("\nCategorical columns - basic stats:")
    print(df.describe())

    print("\nMemory usage:")
    print(df.memory_usage(deep=True))

    print("\nFirst 5 rows:")
    print(df.head())


In [26]:
inspect_data(pd.read_csv(RAW_DATA_PATH))

Number of rows: 251079
Number of columns: 15

Column names:
['Unnamed: 0', 'brand', 'model', 'color', 'registration_date', 'year', 'price_in_euro', 'power_kw', 'power_ps', 'transmission_type', 'fuel_type', 'fuel_consumption_l_100km', 'fuel_consumption_g_km', 'mileage_in_km', 'offer_description']

Data types:
Unnamed: 0                    int64
brand                           str
model                           str
color                           str
registration_date               str
year                            str
price_in_euro                   str
power_kw                        str
power_ps                        str
transmission_type               str
fuel_type                       str
fuel_consumption_l_100km        str
fuel_consumption_g_km           str
mileage_in_km               float64
offer_description               str
dtype: object

Missing values:
                          count  percent
Unnamed: 0                    0     0.00
brand                         0     0

In [34]:
def preprocess_data(df):
    data = df.copy()
    rows_start = len(data)
    print(f"Početni broj redova: {rows_start}")

    # ---------------------------------------------------------------
    # 1. DROPOVANJE NEPOTREBNIH KOLONA
    # Unnamed: 0 je stari CSV index, offer_description je slobodan tekst
    cols_to_drop = ["Unnamed: 0", "offer_description"]
    for col in cols_to_drop:
        if col in data.columns:
            data = data.drop(columns=[col])
            print(f"Dropovana kolona: {col}")

    # ---------------------------------------------------------------
    # 2. KONVERZIJA U NUMERIČKE TIPOVE
    # Kolone kao "10,9 l/100 km" ili "- (g/km)" se čiste i pretvaraju u float
    numeric_columns = [
        "year",
        "price_in_euro",
        "power_kw",
        "power_ps",
        "fuel_consumption_l_100km",
        "fuel_consumption_g_km",
        "mileage_in_km",
    ]
    for col in numeric_columns:
        if col in data.columns:
            data[col] = fix_numeric_column(data[col])

    # ---------------------------------------------------------------
    # 3. ČIŠĆENJE KATEGORIČKIH KOLONA
    # fuel_type ima 136 unikatnih vrednosti - normalizujemo na osnovne tipove
    if "fuel_type" in data.columns:
        data["fuel_type"] = data["fuel_type"].str.strip().str.lower()

        fuel_map = {
            "petrol": "petrol",
            "benzin": "petrol",
            "gasoline": "petrol",
            "diesel": "diesel",
            "electric": "electric",
            "electric/gasoline": "hybrid",
            "electric/diesel": "hybrid",
            "hybrid": "hybrid",
            "lpg": "lpg",
            "cng": "cng",
            "hydrogen": "hydrogen",
        }
        data["fuel_type"] = data["fuel_type"].map(fuel_map).fillna("other")

    # Normalizacija ostalih kategoričkih kolona
    for col in ["brand", "model", "color", "transmission_type"]:
        if col in data.columns:
            data[col] = data[col].str.strip().str.lower()

    # ---------------------------------------------------------------
    # 4. FEATURE ENGINEERING
    # ---------------------------------------------------------------
    # Parsiranje registration_date u mesec i godinu, pa drop originala
    if "registration_date" in data.columns:
        reg = pd.to_datetime(data["registration_date"], format="%m/%Y", errors="coerce")
        data["registration_month"] = reg.dt.month
        data["registration_year"] = reg.dt.year
        data = data.drop(columns=["registration_date"])

    # Starost automobila
    if "year" in data.columns:
        current_year = datetime.now().year
        data["car_age"] = current_year - data["year"]
        data["car_age"] = data["car_age"].clip(lower=0)

    # power_kw i power_ps su linearno zavisni (1 kW ≈ 1.36 PS)
    # Popunjavamo kw iz ps gde fali, pa dropujemo ps
    if "power_kw" in data.columns and "power_ps" in data.columns:
        mask = data["power_kw"].isna() & data["power_ps"].notna()
        data.loc[mask, "power_kw"] = data.loc[mask, "power_ps"] / 1.36
        data = data.drop(columns=["power_ps"])
        print("power_ps dropovan (redundantan sa power_kw)")

    # ---------------------------------------------------------------
    # 5. UKLANJANJE OUTLIERA (postavljamo na NaN, ne brišemo redove)
    # ---------------------------------------------------------------
    outlier_rules = {
        "mileage_in_km": (0, 1_000_000),        # max 1M km je realno
        "price_in_euro": (0, 1_000_000),         # max 1M eur
        "power_kw": (0, 1000),                   # max ~1000 kW (Bugatti nivo)
        "fuel_consumption_l_100km": (0, 50),     # max 50 l/100km
    }

    if "year" in data.columns:
        current_year = datetime.now().year
        outlier_rules["year"] = (1900, current_year + 1)

    for col, (low, high) in outlier_rules.items():
        if col in data.columns:
            outliers = (data[col] < low) | (data[col] > high)
            count = outliers.sum()
            if count > 0:
                data.loc[outliers, col] = np.nan
                print(f"Outlieri u {col}: {count} vrednosti postavljeno na NaN")

    # ---------------------------------------------------------------
    # 6. FILTRIRANJE NEVALIDNIH REDOVA
    # ---------------------------------------------------------------
    # Cena mora postojati i biti pozitivna - bez nje nema smisla za analizu
    if "price_in_euro" in data.columns:
        before_price = len(data)
        data = data[data["price_in_euro"].notna() & (data["price_in_euro"] > 0)]
        print(f"Uklonjeno {before_price - len(data)} redova bez validne cene")

    # ---------------------------------------------------------------
    # 7. UKLANJANJE DUPLIKATA
    # ---------------------------------------------------------------
    before_dup = len(data)
    data = data.drop_duplicates()
    print(f"Uklonjeno {before_dup - len(data)} duplikata")

    # ---------------------------------------------------------------
    # 8. POPUNJAVANJE MISSING VREDNOSTI
    # ---------------------------------------------------------------
    # Numeričke: median (robustan na outliere)
    # Kategoričke: mode (najčešća vrednost)
    for col in data.columns:
        missing = data[col].isna().sum()
        if missing == 0:
            continue

        if pd.api.types.is_numeric_dtype(data[col]):
            median_val = data[col].median()
            data[col] = data[col].fillna(median_val)
            print(f"  {col}: {missing} NaN popunjeno medijanom ({median_val:.2f})")
        else:
            mode_val = data[col].mode(dropna=True)
            if not mode_val.empty:
                fill = mode_val.iloc[0]
                data[col] = data[col].fillna(fill)
                print(f"  {col}: {missing} NaN popunjeno modom ('{fill}')")
            else:
                data[col] = data[col].fillna("unknown")
                print(f"  {col}: {missing} NaN popunjeno sa 'unknown'")

    # ---------------------------------------------------------------
    # 9. LOGOVANJE REZULTATA
    # ---------------------------------------------------------------
    print(f"\nKrajnji broj redova: {len(data)}")
    print(f"Ukupno uklonjeno: {rows_start - len(data)} redova ({(rows_start - len(data)) / rows_start * 100:.2f}%)")
    print(f"Preostali NaN: {data.isnull().sum().sum()}")
    print(f"Finalne kolone: {data.columns.tolist()}")
    print(f"Finalni dtypes:\n{data.dtypes}")

    return data

In [37]:
def main():
    print(f"Loading data from: {RAW_DATA_PATH}")
    raw_data = pd.read_csv(RAW_DATA_PATH)
    # run preprocessing
    processed_data = preprocess_data(raw_data)

    # save to csv
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    processed_data.to_csv(PROCESSED_DATA_PATH, index=False)

    print(f"Saved to: {PROCESSED_DATA_PATH}")
    print(f"Rows: {len(processed_data)}")
    print(f"Columns: {len(processed_data.columns)}")
    print(f"Any nulls left: {processed_data.isnull().sum().sum()}")


In [38]:
main()

Loading data from: D:\Work\used-car-price-prediction\DataSet\Raw\data.csv
Početni broj redova: 251079
Dropovana kolona: Unnamed: 0
Dropovana kolona: offer_description
power_ps dropovan (redundantan sa power_kw)
Outlieri u mileage_in_km: 31 vrednosti postavljeno na NaN
Outlieri u price_in_euro: 16 vrednosti postavljeno na NaN
Outlieri u power_kw: 53 vrednosti postavljeno na NaN
Outlieri u fuel_consumption_l_100km: 1755 vrednosti postavljeno na NaN
Outlieri u year: 67 vrednosti postavljeno na NaN
Uklonjeno 181 redova bez validne cene
Uklonjeno 7097 duplikata
  color: 84 NaN popunjeno modom ('black')
  year: 33 NaN popunjeno medijanom (2018.00)
  power_kw: 131 NaN popunjeno medijanom (110.00)
  fuel_consumption_l_100km: 28187 NaN popunjeno medijanom (5.70)
  fuel_consumption_g_km: 36408 NaN popunjeno medijanom (135.00)
  mileage_in_km: 94 NaN popunjeno medijanom (69379.00)
  registration_month: 33 NaN popunjeno medijanom (6.00)
  registration_year: 33 NaN popunjeno medijanom (2018.00)
  c

In [39]:
df = pd.read_csv(PROCESSED_DATA_PATH)
print(inspect_data(df))

Number of rows: 243801
Number of columns: 14

Column names:
['brand', 'model', 'color', 'year', 'price_in_euro', 'power_kw', 'transmission_type', 'fuel_type', 'fuel_consumption_l_100km', 'fuel_consumption_g_km', 'mileage_in_km', 'registration_month', 'registration_year', 'car_age']

Data types:
brand                           str
model                           str
color                           str
year                        float64
price_in_euro               float64
power_kw                    float64
transmission_type               str
fuel_type                       str
fuel_consumption_l_100km    float64
fuel_consumption_g_km       float64
mileage_in_km               float64
registration_month          float64
registration_year           float64
car_age                     float64
dtype: object

Missing values:
                          count  percent
brand                         0      0.0
model                         0      0.0
color                         0      0.0
year 

In [43]:
def final_preprocess(df):
    data = df.copy()
    rows_start = len(data)
    print(f"Početni broj redova: {rows_start}\n")

    # ---------------------------------------------------------------
    # 1. UKLANJANJE DUPLIKATA (52 reda)
    before = len(data)
    data = data.drop_duplicates()
    print(f"[Duplikati] Uklonjeno: {before - len(data)}")

    # ---------------------------------------------------------------
    # 2. REDUNDANTNE KOLONE
    # year == registration_year (100% identične), dropuj jednu
    # car_age je izvedena iz year, takodje redundantna
    cols_to_drop = []

    if "registration_year" in data.columns and "year" in data.columns:
        cols_to_drop.append("registration_year")

    if "car_age" in data.columns:
        cols_to_drop.append("car_age")

    if cols_to_drop:
        data = data.drop(columns=cols_to_drop)
        print(f"[Redundantne kolone] Dropovane: {cols_to_drop}")

    # Ponovo izračuinavanje car_age kolone
    if "year" in data.columns:
        current_year = 2026
        data["car_age"] = current_year - data["year"]
        # Nevalidne vrednosti car_age (negativne ili previsoke)
        invalid_age = (data["car_age"] < 0) | (data["car_age"] > 50)
        print(f"[car_age] Nevalidnih: {invalid_age.sum()} -> uklanjam redove")
        data = data[~invalid_age]

    # ---------------------------------------------------------------
    # 3. OUTLIER FILTRIRANJE
    # Cene ispod 500€ su verovatno oglasi bez realne cene donji outlier
    # Cene iznad 300k€ su ultra-luksuzni auti koji kvare regresiju gornji outlier
    filters = {
        "price_in_euro": (500, 300_000),
        "power_kw": (20, 600),                    # <20kW nerealno za auto
        "fuel_consumption_l_100km": (4, 30),       # 0 je greška, >30 outlier,a i sve manje od 4l po 100km je sumnjivo
        "mileage_in_km": (0, 500_000),             # >500k sumnjivo
    }

    for col, (low, high) in filters.items():
        if col in data.columns:
            before = len(data)
            data = data[(data[col] >= low) & (data[col] <= high)]
            removed = before - len(data)
            if removed > 0:
                print(f"[Outlieri] {col} ({low}-{high}): uklonjeno {removed} redova")

    # ---------------------------------------------------------------
    # 4. ČIŠĆENJE KATEGORIČKIH KOLONA
    # transmission_type "unknown" -> dropuj te redove (mali broj)
    if "transmission_type" in data.columns:
        unknown_mask = data["transmission_type"] == "unknown"
        print(f"[Transmission] 'unknown' redova: {unknown_mask.sum()} -> uklanjam")
        data = data[~unknown_mask]

        # semi-automatic ima samo ~315 redova, spoji sa automatic
        data["transmission_type"] = data["transmission_type"].replace(
            "semi-automatic", "automatic"
        )

    # Retki modeli (manje od 50 redova) -> zameni sa "other"
    if "model" in data.columns:
        model_counts = data["model"].value_counts()
        rare_models = model_counts[model_counts < 50].index
        data["model"] = data["model"].where(~data["model"].isin(rare_models), "other")
        print(f"[Model] {len(rare_models)} retkih modela zamenjeno sa 'other'")
        print(f"[Model] Preostalih unikatnih: {data['model'].nunique()}")

    # Retki brendovi (manje od 100 redova) -> zameni sa "other"
    if "brand" in data.columns:
        brand_counts = data["brand"].value_counts()
        rare_brands = brand_counts[brand_counts < 100].index
        if len(rare_brands) > 0:
            data["brand"] = data["brand"].where(~data["brand"].isin(rare_brands), "other")
            print(f"[Brand] {len(rare_brands)} retkih brendova zamenjeno sa 'other'")

    # ---------------------------------------------------------------
    # 5. MULTIKOLINEARNOST CHECK
    # year i car_age su savršeno korelisani -> zadrži samo car_age
    if "year" in data.columns and "car_age" in data.columns:
        data = data.drop(columns=["year"])
        print("[Multikolinearnost] Dropovana 'year' (redundantna sa car_age)")

    # fuel_consumption_g_km i fuel_consumption_l_100km su visoko korelisani
    if "fuel_consumption_g_km" in data.columns and "fuel_consumption_l_100km" in data.columns:
        corr = data["fuel_consumption_g_km"].corr(data["fuel_consumption_l_100km"])
        print(f"[Multikolinearnost] Korelacija g_km vs l_100km: {corr:.3f}")
        if abs(corr) > 0.8:
            data = data.drop(columns=["fuel_consumption_g_km"])
            print("  -> Dropovana 'fuel_consumption_g_km'")

    # ---------------------------------------------------------------
    # 6. KRAJ!!!
    print(f"Krajnji broj redova: {len(data)}")
    print(f"Ukupno uklonjeno: {rows_start - len(data)} ({(rows_start - len(data))/rows_start*100:.1f}%)")
    print(f"Broj kolona: {len(data.columns)}")
    print(f"Kolone: {data.columns.tolist()}")
    print(f"Missing vrednosti: {data.isnull().sum().sum()}")
    print(f"\nNumeričke kolone - statistika:")
    print(data.describe().round(2))

    return data

In [44]:
def final():
    # run preprocessing
    processed_data = final_preprocess(df)

    # save to csv
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    processed_data.to_csv(PROCESSED_DATA_PATH, index=False)

    print(f"Saved to: {PROCESSED_DATA_PATH}")
    print(f"Rows: {len(processed_data)}")
    print(f"Columns: {len(processed_data.columns)}")
    print(f"Any nulls left: {processed_data.isnull().sum().sum()}")

In [45]:
final()

Početni broj redova: 243801

[Duplikati] Uklonjeno: 52
[Redundantne kolone] Dropovane: ['registration_year', 'car_age']
[car_age] Nevalidnih: 0 -> uklanjam redove
[Outlieri] price_in_euro (500-300000): uklonjeno 704 redova
[Outlieri] power_kw (20-600): uklonjeno 74 redova
[Outlieri] fuel_consumption_l_100km (4-30): uklonjeno 10454 redova
[Outlieri] mileage_in_km (0-500000): uklonjeno 99 redova
[Transmission] 'unknown' redova: 1084 -> uklanjam
[Model] 732 retkih modela zamenjeno sa 'other'
[Model] Preostalih unikatnih: 559
[Brand] 4 retkih brendova zamenjeno sa 'other'
[Multikolinearnost] Dropovana 'year' (redundantna sa car_age)
[Multikolinearnost] Korelacija g_km vs l_100km: 0.622
Krajnji broj redova: 231334
Ukupno uklonjeno: 12467 (5.1%)
Broj kolona: 12
Kolone: ['brand', 'model', 'color', 'price_in_euro', 'power_kw', 'transmission_type', 'fuel_type', 'fuel_consumption_l_100km', 'fuel_consumption_g_km', 'mileage_in_km', 'registration_month', 'car_age']
Missing vrednosti: 0

Numeričke 